# **🇨🇦 Canadian Health Outcomes & Physical Activity Analysis**

**Theme**: Health Focused<br />
**Author**: Edwin Ronald Lambert

## 1. Project Overview

This analysis explores the relationship between **Physical Activity (PA)** and various **Health Outcomes** (BMI/Perceived Mental/General Health) among Canadians. Using the **Canadian Community Health Survey (CCHS)** Public Use Microdata File (PUMF), I aim to move beyond simple correlations and understand how socio-economic factors (Income, Education) and behavioral factors (Eating Habits) influence this relationship.

The goal of this project is to answer:
> Does a higher frequency of physical activity strictly correlate with better health metrics across all income levels and age groups, or do sedentary behaviors and diet play a larger confounding role?

## 2. The Dataset: CCHS PUMF

The **Canadian Community Health Survey (CCHS)** is a cross-sectional survey collected by Statistics Canada. It gathers information related to health status, health care utilization, and health determinants for the Canadian population.

- **Source:** Statistics Canada
- **Target Population:** Persons aged 12 and over living in the ten provinces and three territories. Excludes full-time members of the Canadian Forces and persons living on reserves.

## 3. Variable Selection & Dictionary
To ensure a robust analysis of the **Socio-Economic and Mental Drivers** of health, I have selected **12 key variables** with high data completeness (>90%), categorized by their role in the analysis.

| Role | Variable Code | Description | Data Type | Source File |
| :--- | :--- | :--- | :--- | :--- |
| **Outcome (KPIs)** | `HWTDGBCC` | **BMI Category (Adjusted)**<br>*(Underweight, Normal, Overweight, Obese)* | Categorical | PUMF (Pg 20) |
| | `GEN_01` | **Perceived General Health**<br>*(1=Excellent, 2=Very Good... 5=Poor)* | Ordinal | Data Dictionary (Pg 16) |
| | `GEN_05` | **Perceived Mental Health**<br>*(1=Excellent... 5=Poor)* | Ordinal | Data Dictionary (Pg 16) |
| **Drivers** | `GEN_10` | **Perceived Life Stress**<br>*(1=Not at all... 5=Extremely stressful)* | Ordinal | Data Dictionary (Pg 17) |
| **Controls** | `CCCDGCAR` | **Heart Disease**<br>*(1=Yes, 2=No - Health Control)* | Categorical | Grouped Vars (Pg 36) |
| | `CCCDGSKL` | **Musculoskeletal/Back Issues**<br>*(1=Yes, 2=No - Physical Limitation)* | Categorical | Grouped Vars (Pg 35) |
| **Context** | `INCDGRPR` | **Household Income Quintile**<br>*(1=Lowest, 5=Highest)* | Ordinal | Data Dictionary (Pg 112) |
| | `EDDVH3` | **Education Level**<br>*(Sec, Post-Sec, Degree)* | Categorical | Data Dictionary (Pg 13) |
| | `DHHGAGE` | **Age Group**<br>*(12-17, 18-34, 35-49, etc.)* | Categorical | Data Dictionary (Pg 14) |
| | `GEOGPRV` | **Province**<br>*(e.g., 59 = British Columbia)* | Nominal | Data Dictionary (Pg 11) |
| | `DHH_SEX` | **Sex**<br>*(1=Male, 2=Female)* | Nominal | Data Dictionary (Pg 14) |
| **Weight** | `WTS_M` | **Master Survey Weight**<br>*(MANDATORY for population estimates)* | Numerical | Data Dictionary (Pg 113) |

## 4. Exploratory Analysis Objectives
Due to low response rates in optional modules (Diet, Physical Activity, Screen Time), this analysis pivots to focus on the **Socio-Economic, Psychological, and Medical drivers** of health, where data coverage is excellent (>90%).

I will structure the analysis into four targeted investigations:

**1. The "Stress-Obesity" Link**
* **Goal:** Investigate if high psychological stress manifests as physical weight gain.
* **Hypothesis:** Individuals reporting "Extremely Stressful" lives (`GEN_10`) will have a higher prevalence of **Obesity** (`HWTDGBCC`) than those with low stress, potentially indicating stress-eating or cortisol-induced weight gain as a population-level trend.
* **Variables:** `GEN_10` (Life Stress), `HWTDGBCC` (Adjusted BMI).

**2. The "Wealth-Health" Gradient**
* **Goal:** Quantify the protective effect of income on general well-being.
* **Hypothesis:** Higher **Income Quintiles** (`INCDGRPR`) correlate strongly with better **Perceived General Health** (`GEN_01`) and lower rates of **Heart Disease** (`CCCDGCAR`). We expect a "step-ladder" effect where every increase in income tier yields a measurable improvement in health.
* **Variables:** `INCDGRPR` (Income), `GEN_01` (General Health), `CCCDGCAR` (Heart Disease).

**3. The "Silent Suffering" (Pain vs. Mental Health)**
* **Goal:** Analyze the hidden mental toll of chronic physical limitations.
* **Hypothesis:** Respondents with **Chronic Back Problems** (`CCCDGSKL`) will report significantly lower **Mental Health** scores (`GEN_05`) than the general population. This challenges the medical model by framing chronic pain as a mental health issue as much as a physical one.
* **Variables:** `CCCDGSKL` (Back Problems), `GEN_05` (Mental Health).

**4. The Demographic Baseline (Age & Geography)**
* **Goal:** Determine if health outcomes are determined by *who* you are or *where* you live.
* **Hypothesis:** While **General Health** (`GEN_01`) naturally declines with **Age** (`DHHGAGE`), we hypothesize that certain **Provinces** (`GEOGPRV`) will show higher resilience (better health scores at older ages) due to regional socio-economic factors.
* **Variables:** `DHHGAGE` (Age), `GEOGPRV` (Province), `GEN_01` (Health).

---

**Synthesis: The "Big Picture"**

After these deep dives, I will combine the strongest drivers into a **multivariate summary**:
1.  **Driver Analysis:** Which is the stronger predictor of Poor Health? Is it **Low Income** or **High Stress**?
2.  **Profile Creation:** I will define the **"Vulnerable Profile"** (e.g., "Low Income, High Stress, Chronic Pain") vs. the **"Resilient Profile"** to inform public health targeting.

In [11]:
# LIBRARY IMPORTS & DATA LOADING

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# 1. Configurations
# Set pandas to display all columns to avoid truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

# Use a clean plotting style
sns.set_theme(style="whitegrid")

# 2. Load Data
file_path = '../data/raw/pumf_cchs.csv'

if os.path.exists(file_path):
    try:
        # Load the full dataset
        df_raw = pd.read_csv(file_path, low_memory=False)
        print(f"Dataset Successfully Loaded. Found {len(df_raw.columns)} columns.")
        
        # Select relevant columns for analysis
        target_cols = [
            'HWTDGBCC',  # Outcome: BMI (Adjusted)
            'GEN_01',    # Outcome: General Health
            'GEN_05',    # Outcome: Mental Health
            'GEN_10',    # Driver: Life Stress
            'CCCDGCAR',  # Medical: Heart Disease
            'CCCDGSKL',  # Medical: Back Problems
            'INCDGRPR',  # Socio-Economic: Income
            'EDDVH3',    # Socio-Economic: Education
            'DHHGAGE',   # Demographics: Age
            'GEOGPRV',   # Demographics: Province
            'DHH_SEX',   # Demographics: Sex
            'WTS_M'      # Weight
        ]

        available_cols = [col for col in target_cols if col in df_raw.columns]
        missing_cols = list(set(target_cols) - set(available_cols))
        
        if missing_cols:
            print(f"Warning: The following target columns are missing from the dataset: {missing_cols}")
            
        df = df_raw[available_cols].copy()

        print(f"Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")

        # 3. Rename for Readability
        # Mapping of original column names to more descriptive names
        column_mapping = {
            'HWTDGBCC': 'BMI_Category',
            'GEN_01':   'Perceived_Health',
            'GEN_05':   'Perceived_Mental_Health',
            'GEN_10':   'Life_Stress',
            'CCCDGCAR': 'Has_Heart_Disease',
            'CCCDGSKL': 'Has_Back_Problems',
            'INCDGRPR': 'Income_Quintile',
            'EDDVH3':   'Education_Level',
            'DHHGAGE':  'Age_Group',
            'GEOGPRV':  'Province',
            'DHH_SEX':  'Sex',
            'WTS_M':    'Weight'
        }

        df_renamed = df.rename(columns=column_mapping)

        # 4. Preview the Data
        display(df_renamed.head())

    except Exception as e:
        print(f"Error during processing: {e}")

else:
    print("Error: File not found. Please ensure file is in the 'data/raw/' directory.")

Dataset Successfully Loaded. Found 255 columns.
Dimensions: 67,079 rows x 12 columns


,BMI_Category,Perceived_Health,Perceived_Mental_Health,Life_Stress,Has_Heart_Disease,Has_Back_Problems,Income_Quintile,Education_Level,Age_Group,Province,Sex,Weight
0,1,3,3,1,2,2,2,2,5,35,1,215.62
1,2,3,3,3,2,1,5,3,4,24,2,338.32
2,6,3,4,4,6,6,9,3,1,35,2,189.70
3,1,2,3,3,6,2,5,3,2,59,2,145.95
4,1,3,4,3,9,9,4,3,3,24,1,355.66


In [13]:
# CLEANING & PREPROCESSING

# 1. Define Standard "Not Stated" Codes to Filter Out
# We replace these codes with NaN for easier handling
nan_codes = [6, 7, 8, 9,
             96, 97, 98, 99,
             99.6, 99.9,
             999.6, 999.9
             ]

# 2. Creating a working copy to avoid modifying the original data
df_clean = df_renamed.copy()

# 3. Replace "Valid Skip"/"Not Stated" Codes with NaN
cols_to_clean = [
    'BMI_Category',
    'Perceived_Health',
    'Perceived_Mental_Health',
    'Life_Stress',
    'Has_Heart_Disease',
    'Has_Back_Problems',
    'Income_Quintile',
    'Education_Level',
    'Age_Group'
]

for col in cols_to_clean:
    # Ensure numeric types
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    # Replace the specific nan_codes with numeric NaN
    df_clean[col] = df_clean[col].replace(nan_codes, np.nan)


# 4. Label Mapping (Converting Codes to Descriptive Labels)
# Based on the CCHS Data Dictionary
labels_bmi_category = {
    1: 'Underweight / Normal Weight',
    2: 'Overweight / Obese'
}

labels_perceived_health = {
    1: 'Excellent',
    2: 'Very Good',
    3: 'Good',
    4: 'Fair',
    5: 'Poor'
}

labels_perceived_mental_health = {
    1: 'Excellent',
    2: 'Very Good',
    3: 'Good',
    4: 'Fair',
    5: 'Poor'
}

labels_life_stress = {
    1: 'Not at all stressful',
    2: 'Not very stressful',
    3: 'A bit stressful',
    4: 'Quite a bit stressful',
    5: 'Extremely stressful'
}

labels_has_heart_disease = {
    1: 'Yes',
    2: 'No'
}

labels_has_back_problems = {
    1: 'Yes',
    2: 'No' 
}

labels_income_quintile = {
    1: 'Lowest',
    2: 'Lower Middle',
    3: 'Middle',
    4: 'Upper Middle',
    5: 'Highest'
}

labels_education_level = {
    1: 'Less than Secondary',
    2: 'Secondary Graduate',
    3: 'Post-Secondary/Degree'
}

labels_age_group = {
    1: '12-17 years',
    2: '18-34 years',
    3: '35-49 years',
    4: '50-64 years',
    5: '65+ years'
}

labels_province = {
    10: 'Newfoundland and Labrador',
    11: 'Prince Edward Island',
    12: 'Nova Scotia',
    13: 'New Brunswick',
    24: 'Quebec',
    35: 'Ontario',
    46: 'Manitoba',
    47: 'Saskatchewan',
    48: 'Alberta',
    59: 'British Columbia',
    60: 'Yukon',
    61: 'Northwest Territories',
    62: 'Nunavut'
}

labels_sex = {
    1: 'Male',
    2: 'Female'
}

# Apply Mappings
df_clean['BMI_Category'] = df_clean['BMI_Category'].map(labels_bmi_category)
df_clean['Perceived_Health'] = df_clean['Perceived_Health'].map(labels_perceived_health)
df_clean['Perceived_Mental_Health'] = df_clean['Perceived_Mental_Health'].map(labels_perceived_mental_health)
df_clean['Life_Stress'] = df_clean['Life_Stress'].map(labels_life_stress)
df_clean['Has_Heart_Disease'] = df_clean['Has_Heart_Disease'].map(labels_has_heart_disease)
df_clean['Has_Back_Problems'] = df_clean['Has_Back_Problems'].map(labels_has_back_problems)
df_clean['Income_Quintile'] = df_clean['Income_Quintile'].map(labels_income_quintile)
df_clean['Education_Level'] = df_clean['Education_Level'].map(labels_education_level)
df_clean['Age_Group'] = df_clean['Age_Group'].map(labels_age_group)
df_clean['Province'] = df_clean['Province'].map(labels_province)
df_clean['Sex'] = df_clean['Sex'].map(labels_sex)

# 5. Remove rows where main outcome (BMI) or main driver (Stress) is missing.
initial_count = len(df_clean)
df_clean = df_clean.dropna(subset=['BMI_Category', 'Life_Stress'])
final_count = len(df_clean)

print("Data Cleaning Complete.")
print(f"Removed {initial_count - final_count:,} rows with missing BMI or Life Stress data.")
print(f"Final Analysis Dataset Dimensions: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns.")

# Preview Cleaned Data
display(df_clean[['BMI_Category', 'Life_Stress', 'Income_Quintile', 'Education_Level']].sample(5))

Data Cleaning Complete.
Removed 7,176 rows with missing BMI or Life Stress data.
Final Analysis Dataset Dimensions: 59,903 rows x 12 columns.


,BMI_Category,Life_Stress,Income_Quintile,Education_Level
31087,Underweight / Normal Weight,A bit stressful,Lowest,Post-Secondary/Degree
48850,Overweight / Obese,Quite a bit stressful,Upper Middle,Post-Secondary/Degree
9360,Overweight / Obese,Quite a bit stressful,Middle,Post-Secondary/Degree
51465,Overweight / Obese,A bit stressful,Lowest,NaN
7862,Overweight / Obese,Quite a bit stressful,Middle,Post-Secondary/Degree
